In [11]:
import warnings
import pandas as pd
warnings.filterwarnings('ignore')
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import matplotlib.pyplot as plt
import numpy as np

In [2]:
sentiment_df = pd.read_csv('sentiment_score_40000.csv')
sentiment_df

,title,seendate,Positive,Negative,Neutral,discrete_score,continuous_score
0,Happy International Trans Day of Visibility | ...,2021-03-31,0.113769,0.019423,0.866807,0,0.094346
1,Chain of Events Update from Blockchain . com R...,2021-03-31,0.572007,0.009366,0.418627,1,0.562640
2,"CME Develops Micro Bitcoin Futures , Set to La...",2021-03-31,0.074450,0.009746,0.915804,0,0.064704
3,Daily Briefing : Archegos Leveraged Blow - Up ...,2021-03-31,0.036394,0.122330,0.841276,0,-0.085936
4,"Trekkies Rejoice , Real World Shatner NFTs Now...",2021-03-31,0.398860,0.013678,0.587461,0,0.385182
...,...,...,...,...,...,...,...
39995,Ratchet & Clank : Rift Apart Review - Two Lomb...,2021-06-08,0.146859,0.029227,0.823914,0,0.117632
39996,Bands line up for Otley Live Music Festival,2021-06-08,0.036507,0.028912,0.934581,0,0.007594
39997,"To prevent delirium , increase mobility , conn...",2021-06-08,0.116923,0.011520,0.871558,0,0.105403
39998,We Found A Dog Backpack That Literally Does It...,2021-06-08,0.056575,0.015015,0.928409,0,0.041560


In [8]:
raw_df = sentiment_df.iloc[1:300,[0,1,5,6]]
raw_df

,title,seendate,discrete_score,continuous_score
1,Chain of Events Update from Blockchain . com R...,2021-03-31,1,0.562640
2,"CME Develops Micro Bitcoin Futures , Set to La...",2021-03-31,0,0.064704
3,Daily Briefing : Archegos Leveraged Blow - Up ...,2021-03-31,0,-0.085936
4,"Trekkies Rejoice , Real World Shatner NFTs Now...",2021-03-31,0,0.385182
5,"Cards , Games , And NFTs : An Interview With A...",2021-03-31,0,0.013549
...,...,...,...,...
295,Markets in first - quarter : Riding a tiger an...,2021-03-31,0,0.009673
296,Business Highlights,2021-03-31,0,-0.003789
297,Markets in first quarter : Riding a tiger and ...,2021-03-31,0,0.079439
298,Markets in first - quarter : Riding a tiger an...,2021-03-31,0,0.051685


In [9]:
raw_df['ground_truth'] = 0
raw_df

,title,seendate,discrete_score,continuous_score,ground_truth
1,Chain of Events Update from Blockchain . com R...,2021-03-31,1,0.562640,0
2,"CME Develops Micro Bitcoin Futures , Set to La...",2021-03-31,0,0.064704,0
3,Daily Briefing : Archegos Leveraged Blow - Up ...,2021-03-31,0,-0.085936,0
4,"Trekkies Rejoice , Real World Shatner NFTs Now...",2021-03-31,0,0.385182,0
5,"Cards , Games , And NFTs : An Interview With A...",2021-03-31,0,0.013549,0
...,...,...,...,...,...
295,Markets in first - quarter : Riding a tiger an...,2021-03-31,0,0.009673,0
296,Business Highlights,2021-03-31,0,-0.003789,0
297,Markets in first quarter : Riding a tiger and ...,2021-03-31,0,0.079439,0
298,Markets in first - quarter : Riding a tiger an...,2021-03-31,0,0.051685,0


In [10]:
raw_df.to_csv('finbert_300.csv', index=False)

In [2]:
label_df = pd.read_csv('finbert_300.csv')
label_df

,title,seendate,discrete_score,continuous_score,ground_truth
0,Chain of Events Update from Blockchain . com R...,2021-03-31,1,0.562640,1
1,"CME Develops Micro Bitcoin Futures , Set to La...",2021-03-31,0,0.064704,1
2,Daily Briefing : Archegos Leveraged Blow - Up ...,2021-03-31,0,-0.085936,-1
3,"Trekkies Rejoice , Real World Shatner NFTs Now...",2021-03-31,0,0.385182,1
4,"Cards , Games , And NFTs : An Interview With A...",2021-03-31,0,0.013549,0
...,...,...,...,...,...
294,Markets in first - quarter : Riding a tiger an...,2021-03-31,0,0.009673,0
295,Business Highlights,2021-03-31,0,-0.003789,0
296,Markets in first quarter : Riding a tiger and ...,2021-03-31,0,0.079439,0
297,Markets in first - quarter : Riding a tiger an...,2021-03-31,0,0.051685,0


In [13]:
sum = (label_df['discrete_score'] == label_df['ground_truth']).sum()
accuracy_discrete = round(sum/len(label_df),2)
accuracy_discrete

0.59

In [12]:
conditions = [
    label_df['continuous_score'] > 0.05,
    label_df['continuous_score'].between(-0.05, 0.05, inclusive="neither"),
    label_df['continuous_score'] < -0.05
]

choices = [1, 0, -1]

label_df['continuous_label'] = np.select(conditions, choices)
label_df

,title,seendate,discrete_score,continuous_score,ground_truth,continuous_label
0,Chain of Events Update from Blockchain . com R...,2021-03-31,1,0.562640,1,1
1,"CME Develops Micro Bitcoin Futures , Set to La...",2021-03-31,0,0.064704,1,1
2,Daily Briefing : Archegos Leveraged Blow - Up ...,2021-03-31,0,-0.085936,-1,-1
3,"Trekkies Rejoice , Real World Shatner NFTs Now...",2021-03-31,0,0.385182,1,1
4,"Cards , Games , And NFTs : An Interview With A...",2021-03-31,0,0.013549,0,0
...,...,...,...,...,...,...
294,Markets in first - quarter : Riding a tiger an...,2021-03-31,0,0.009673,0,0
295,Business Highlights,2021-03-31,0,-0.003789,0,0
296,Markets in first quarter : Riding a tiger and ...,2021-03-31,0,0.079439,0,1
297,Markets in first - quarter : Riding a tiger an...,2021-03-31,0,0.051685,0,1


In [14]:
sum = (label_df['continuous_label'] == label_df['ground_truth']).sum()
accuracy_continuous = round(sum/len(label_df),2)
accuracy_continuous

0.67

### The accuracy of FinBERT has a 59% accuracy on discrete_score, and a 67% accuracy on continuous_label.
### The next step is to use the retrained finbert on this finbert_300 dataset and check its accuracy.

### Use the last 10 days sentiment to make decisions